In [ ]:
!pip install qiskit
!pip install qiskit-aer
!pip install qiskit-ibm-runtime

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EnvironmentOptions, EstimatorOptions, SamplerOptions
from qiskit_aer import AerSimulator
import numpy as np
from numpy import pi
from matplotlib import pyplot as plt
import matplotlib
from scipy.optimize import minimize
from qiskit.circuit.library import QAOAAnsatz, hamiltonian_variational_ansatz, XXPlusYYGate
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit import Parameter
from scipy.optimize import minimize
from scipy.linalg import eigh
from scipy.special import erf
from functools import partial
from qiskit_aer import AerSimulator
from qiskit.circuit.classical import expr
import random

## Measurement


In [ ]:
def measure_ZZ(qc, q0, q1, cbit):
    qc.cx(q0, q1)
    qc.measure(q1, cbit)
    qc.cx(q0, q1)

In [ ]:
def measure_XI(qc, q0, q1, cbit):
    qc.h(q0)
    qc.measure(q0, cbit)
    qc.h(q0)

**Important: fix convention $ Y = S X S^\dagger$. Later formulas must use the same convention to ensure a correct pauli tracking**

In [ ]:
def measure_YI(qc, q0, q1, cbit):
    qc.sdg(q0)
    measure_XI(qc, q0, q1, cbit)
    qc.s(q0)

In [ ]:
def measure_ZY(qc, q0, q1, cbit):

    # with the same convention, Y = S X S^d = S H Z H S^d
    qc.sdg(q1)
    qc.h(q1)
    measure_ZZ(qc, q0, q1, cbit)
    qc.h(q1)
    qc.s(q1)

## Single qubit clifford group

In [ ]:
def add_S(qc, q, cbit):
    c = cbit

    # (ancilla, data) = (q+1, q)

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_YI(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # measurement bit c_i encodes s_i = (-1)^{c_i}
    # s_i s_j = (-1)^{c[i] + c[j]}
    # product becomes XOR

    # Z^{(1 + s0 s1 s2)/2}
    # exponent = 1 when s0 s1 s2 = +1
    # s0 s1 s2 = (-1)^{c0 + c1 + c2}
    # +1 when (c0 + c1 + c2) mod 2 = 0 (even parity)

    parity = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[2])
    with qc.if_test(expr.logic_not(parity)):
        qc.z(q)

    qc.reset(q+1) # easy to check with gate-based circuit

    return qc

In [ ]:
def add_HSH(qc, q, cbit):
    c = cbit

    measure_XI(qc, q+1, q, c[0])
    measure_ZZ(qc, q+1, q, c[1])
    measure_ZY(qc, q+1, q, c[2])
    measure_XI(qc, q+1, q, c[3])

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    # Y^{(1 - s0 s3)/2} odd parity
    parity_03 = expr.bit_xor(c[0], c[3])
    with qc.if_test(parity_03):
        qc.y(q)

    qc.reset(q+1)

    return qc

In [ ]:
def add_HS(qc, q, cbit):
    c = cbit

    measure_XI(qc, q+1, q, c[0])  # s0
    measure_ZY(qc, q+1, q, c[1])  # s1
    measure_ZZ(qc, q+1, q, c[2])  # s2
    measure_YI(qc, q+1, q, c[3])  # s3
    measure_XI(qc, q+1, q, c[4])  # s4

    # Z^{(1 + s0 s1 s3)/2} even parity
    parity_013 = expr.bit_xor(expr.bit_xor(c[0], c[1]), c[3])
    with qc.if_test(expr.logic_not(parity_013)):
        qc.z(q)

    # X^{(1 + s1 s2)/2} even parity
    parity_12 = expr.bit_xor(c[1], c[2])
    with qc.if_test(expr.logic_not(parity_12)):
        qc.x(q)

    qc.reset(q+1)

    return qc

## Single qubit clifford gate in supremacy circuit

In [ ]:
def add_sqrtX(qc, q, cbit):
    add_HSH(qc, q, cbit)
    return qc

In [ ]:
def add_sqrtY(qc, q, cbit):
    add_S(qc, q, cbit)
    add_HS(qc, q, cbit)
    return qc

In [ ]:
def add_sqrtW(qc, q, cbit):
    qc.tdg(q)
    add_sqrtX(qc, q, cbit)
    qc.t(q)
    return qc

# Random circuit with single qubit gate

qubit geometry: (i,0) = data, (i,1)=ancilla, (i,j)=2i+j

In [ ]:
def random_single_MBQC(N, periods, seed, initial_bitstring):
    random.seed(seed)

    #all circuits with single-qubit clifford gates need at most 5 classical bits
    cbit = ClassicalRegister(5)

    qc = QuantumCircuit(2 * N)
    qc.initialize(Statevector.from_label(initial_bitstring))
    qc.add_register(cbit)

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                add_sqrtX(qc, 2*i, cbit)
            elif gate == "sY":
                add_sqrtY(qc, 2*i, cbit)
            else:
                add_sqrtW(qc, 2*i, cbit)

            last_gate[i] = gate

    return qc

In [ ]:
def random_single_gateQC(N, periods, seed, initial_bitstring):
    random.seed(seed)

    qc = QuantumCircuit(2 * N)
    qc.initialize(Statevector.from_label(initial_bitstring))

    last_gate = [None] * N

    for p in range(periods):
        for i in range(N):

            choices = ["sX", "sY", "sW"]
            if last_gate[i] is not None:
                choices.remove(last_gate[i])
            gate = random.choice(choices)

            # apply
            if gate == "sX":
                qc.sx(2*i)
            elif gate == "sY":
                qc.ry(np.pi/2, 2*i)
            else:
                qc.tdg(2*i)
                qc.sx(2*i)
                qc.t(2*i)

            last_gate[i] = gate

    return qc

# N=2 (4-qubit) circuit test 2-qubit hilbert space
## MBQC and GBQC have final states differ by a global phase

Test on the complete basis  
$$
\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\} \otimes \mathcal{H}_{\text{ancilla}}.
$$
to cover the full Hilbert space. Always reset ancilla qubits to |0>.

This test generalizes to arbitrary $N$ because the circuit with single-qubit gates factorizes as
$$
U_{\text{circuit}}^{(N)} = \bigotimes_{i=1}^N U_{\text{circuit}, i}.
$$
Better choose a large period. I choose 100.

In [ ]:
N=2
seed=1
periods=100
initial_bitstring = "0000"

qc1 = random_single_gateQC(N, periods, seed, initial_bitstring)
sv1 = Statevector.from_instruction(qc1)

print("gate on |00>")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = random_single_MBQC(N, periods, seed, initial_bitstring)
qc2.save_statevector(conditional=True)
sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()
print("MBQC on |00>:")
print(result.data(0)["statevector"])

gate on |00>
[ 0.07821948-0.07760467j  0.3325835 +0.19657532j  0.        +0.j
  0.        +0.j         -0.23129501+0.09790856j -0.53711987-0.69787131j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j        ]
MBQC on |00>:
{'0x1a': Statevector([-0.07821948+0.07760467j, -0.3325835 -0.19657532j,
              0.        +0.j        ,  0.        +0.j        ,
              0.23129501-0.09790856j,  0.53711987+0.69787131j,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
             -0.        +0.j        ,  0.        -0.j        ,
              0.        +0.j        ,  0.        +0.j        ],
            dims=(2, 2, 2, 2))}


In [ ]:
N=2
seed=1
periods=100
initial_bitstring = "0100"

qc1 = random_single_gateQC(N, periods, seed, initial_bitstring)
sv1 = Statevector.from_instruction(qc1)

print("gate on |10>")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = random_single_MBQC(N, periods, seed, initial_bitstring)
qc2.save_statevector(conditional=True)
sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()
print("MBQC on |10>:")
print(result.data(0)["statevector"])

gate on |10>
[ 0.08795558+0.23526003j -0.72014263+0.50686952j  0.        +0.j
  0.        +0.j         -0.00256071+0.11015541j -0.37595782+0.08893458j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j        ]
MBQC on |10>:
{'0x2': Statevector([-0.08795558-0.23526003j,  0.72014263-0.50686952j,
              0.        +0.j        ,  0.        +0.j        ,
              0.00256071-0.11015541j,  0.37595782-0.08893458j,
              0.        +0.j        ,  0.        +0.j        ,
              0.        -0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ],
            dims=(2, 2, 2, 2))}


In [ ]:
N=2
seed=1
periods=100
initial_bitstring = "0001"

qc1 = random_single_gateQC(N, periods, seed, initial_bitstring)
sv1 = Statevector.from_instruction(qc1)

print("gate on |10>")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = random_single_MBQC(N, periods, seed, initial_bitstring)
qc2.save_statevector(conditional=True)
sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()
print("MBQC on |10>:")
print(result.data(0)["statevector"])

gate on |10>
[-0.37595782-0.08893458j  0.00256071+0.11015541j  0.        +0.j
  0.        +0.j          0.72014263+0.50686952j  0.08795558-0.23526003j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j        ]
MBQC on |10>:
{'0x7': Statevector([-0.37595782-0.08893458j,  0.00256071+0.11015541j,
              0.        +0.j        ,  0.        +0.j        ,
              0.72014263+0.50686952j,  0.08795558-0.23526003j,
              0.        +0.j        ,  0.        +0.j        ,
              0.        -0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ],
            dims=(2, 2, 2, 2))}


In [ ]:
N=2
seed=1
periods=100
initial_bitstring = "0101"

qc1 = random_single_gateQC(N, periods, seed, initial_bitstring)
sv1 = Statevector.from_instruction(qc1)

print("gate on |11>")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = random_single_MBQC(N, periods, seed, initial_bitstring)
qc2.save_statevector(conditional=True)
sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()
print("MBQC on |11>:")
print(result.data(0)["statevector"])

gate on |11>
[ 0.53711987-0.69787131j -0.23129501-0.09790856j  0.        +0.j
  0.        +0.j          0.3325835 -0.19657532j -0.07821948-0.07760467j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j          0.        +0.j          0.        +0.j
  0.        +0.j        ]
MBQC on |11>:
{'0xa': Statevector([ 0.53711987-0.69787131j, -0.23129501-0.09790856j,
              0.        +0.j        ,  0.        +0.j        ,
              0.3325835 -0.19657532j, -0.07821948-0.07760467j,
              0.        +0.j        ,  0.        +0.j        ,
             -0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ,
             -0.        +0.j        ,  0.        +0.j        ,
              0.        +0.j        ,  0.        +0.j        ],
            dims=(2, 2, 2, 2))}


## (not need to read) Single-qubit gate test on $|0\rangle$ and $|+\rangle$ to cover the whole Hilbert space of qubit

In [ ]:
# sqrt(X) test — input |0>
# remember sqrt(X) rotates Z to -Y, final state is |-y>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)  # need at most 5 classical bits
qc2.add_register(cbit2)
add_sqrtX(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(X) on |0>:")
print(result.data(0)["statevector"])

direct sqrt(X) on |0>:
[0.5+0.5j 0.5-0.5j 0. +0.j  0. +0.j ]
MBQC sqrt(X) on |0>:
{'0x0': Statevector([0.        +0.70710678j, 0.70710678+0.j        ,
             0.        +0.j        , 0.        +0.j        ],
            dims=(2, 2))}


In [ ]:
# sqrt(X) test — input |+>

# 1) direct sqrt(X)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.sx(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(X) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(X)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
qc2.h(0)
add_sqrtX(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(X) on |+>:")
print(result.data(0)["statevector"])

direct sqrt(X) on |+>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]
MBQC sqrt(X) on |+>:
{'0x6': Statevector([0.5+0.5j, 0.5+0.5j, 0. +0.j , 0. +0.j ],
            dims=(2, 2))}


In [ ]:
# sqrt(Y) test — input |0>
# remember sqrt(Y) rotates Z to X, final state is |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
add_sqrtY(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(Y) on |0>:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |0>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]
MBQC sqrt(Y) on |0>:
{'0x15': Statevector([4.79118720e-16+0.70710678j, 3.22109474e-16+0.70710678j,
             0.00000000e+00-0.j        , 0.00000000e+00-0.j        ],
            dims=(2, 2))}


In [ ]:
# sqrt(Y) test — input |+>

# 1) direct sqrt(Y)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.ry(np.pi/2, 0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(Y) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(Y)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)
qc2.add_register(cbit2)
qc2.h(0)
add_sqrtY(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(Y) on |+>:")
print(result.data(0)["statevector"])

direct sqrt(Y) on |+>:
[8.86511593e-17+0.j 1.00000000e+00+0.j 0.00000000e+00+0.j
 0.00000000e+00+0.j]
MBQC sqrt(Y) on |+>:
{'0xf': Statevector([ 2.77555756e-17-2.77555756e-17j,
             -1.09906472e-15-1.00000000e+00j,
              0.00000000e+00+0.00000000e+00j,
              0.00000000e+00-0.00000000e+00j],
            dims=(2, 2))}


In [ ]:
# sqrt(W) test — input |0>

# 1) direct sqrt(W)
qc1 = QuantumCircuit(2)
qc1.tdg(0)
qc1.sx(0)
qc1.t(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(W) on |0>:")
print(sv1.data)


# 2) MBQC sqrt(W)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)  # need at most 5 classical bits
qc2.add_register(cbit2)
add_sqrtW(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(W) on |0>:")
print(result.data(0)["statevector"])

direct sqrt(W) on |0>:
[0.5       +0.5j 0.70710678+0.j  0.        +0.j  0.        +0.j ]
MBQC sqrt(W) on |0>:
{'0x7': Statevector([1.73191211e-16+0.70710678j, 5.00000000e-01+0.5j       ,
             0.00000000e+00+0.j        , 0.00000000e+00+0.j        ],
            dims=(2, 2))}


In [ ]:
# sqrt(W) test — input |+>

# 1) direct sqrt(W)
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.tdg(0)
qc1.sx(0)
qc1.t(0)

sv1 = Statevector.from_instruction(qc1)

print("direct sqrt(W) on |+>:")
print(sv1.data)


# 2) MBQC sqrt(W)
qc2 = QuantumCircuit(2)
cbit2 = ClassicalRegister(5)  # need at most 5 classical bits
qc2.add_register(cbit2)
qc2.h(0)
add_sqrtW(qc2, 0, cbit2)

qc2.save_statevector(conditional=True)

sim = AerSimulator(method="statevector")
result = sim.run(qc2, shots=1).result()

print("MBQC sqrt(W) on |+>:")
print(result.data(0)["statevector"])

direct sqrt(W) on |+>:
[0.35355339-0.14644661j 0.85355339+0.35355339j 0.        +0.j
 0.        +0.j        ]
MBQC sqrt(W) on |+>:
{'0x0': Statevector([0.35355339+0.14644661j, 0.35355339+0.85355339j,
             0.        +0.j        , 0.        +0.j        ],
            dims=(2, 2))}
